# 1. Prepare reusable TinyStories inputs

**Already prepared TinyStories? Skip this notebook.** Attach your existing preparation
output to notebooks 2 and 3. The smaller AR and NCM generators use the same text
rows, frozen tokenizer and VQ codes as the previous models.


Enable **GPU** and **Internet** in Kaggle settings. This notebook clones the fork,
runs synthetic checks, downloads the official TinyStories assets, prepares shared
AR/NCM token rows, and optionally caches frozen VQ-VAE codes.

All reusable files are saved under **`/kaggle/working/ncm-tinystories/`**.
After execution, save a notebook version with outputs and create a **private Kaggle
Dataset** from that folder. Dataset creation/privacy is a manual Kaggle action;
this notebook does not publish or change account settings.

Push the implementation to your fork before running this notebook. No training is
performed here. Code-cache preparation performs one tokenizer pass over the data.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/kuchidareo/3rd-contribution.git'  # Set the 3rd-contribution repository URL.
REPO_REF = 'main'  # Prefer a full commit containing the baseline implementation.
CHECKOUT = Path('/kaggle/working/3rd-contribution')
REPO = CHECKOUT / '261119-NCM-replicating/ncm'

def run(command, cwd=REPO):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

if not CHECKOUT.exists():
    run(['git', 'clone', REPO_URL, CHECKOUT], cwd='/kaggle/working')
run(['git', 'fetch', 'origin', REPO_REF], cwd=CHECKOUT)
run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=CHECKOUT)
required = [REPO / 'baseline_data/prepare.py', REPO / 'training/train_ar.py',
            REPO / 'config/baseline/tinystories.yaml']
if any(not p.is_file() for p in required):
    raise RuntimeError('This revision lacks the baseline code. Commit/push the local changes, then set REPO_REF.')
os.chdir(REPO)
# Keep all downloads and environment records inside the preparation/training output.
run(['bash', 'scripts/kaggle_setup.sh'])


In [ ]:
# Fail before any dataset download if the model/training regression tests fail.
run([sys.executable, '-m', 'pytest', 'tests', '-q'])


In [ ]:
from omegaconf import OmegaConf
import json
import torch

OUTPUT = Path('/kaggle/working/ncm-tinystories')
OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE_VQ_CODES = True
KEEP_RAW_DOWNLOADS = False  # Remove redundant raw files only after successful verification.
CONFIG = OUTPUT / 'preparation.yaml'

cfg = OmegaConf.load(REPO / 'config/baseline/tinystories.yaml')
cfg.paths.artifacts = str(OUTPUT / 'artifacts/tinystories')
cfg.paths.tokenizer_checkpoint = str(OUTPUT / 'artifacts/tinystories/tokenizer.pt')
cfg.paths.data = str(OUTPUT / 'prepared')
cfg.paths.codes = str(OUTPUT / 'codes')
cfg.device = 'cuda'
# Code assignment is FP32 regardless of training precision.
OmegaConf.save(cfg, CONFIG)
print('Reusable output:', OUTPUT)
print('GPU:', torch.cuda.get_device_name())


In [ ]:
prepared_manifest = OUTPUT / 'prepared/manifest.json'
if not prepared_manifest.exists():
    run([sys.executable, '-m', 'baseline_data.prepare', '--config', CONFIG,
         '--download', '--download-root', OUTPUT,
         '--train', OUTPUT / 'official-data/tinystories/train_bpe_8192.bin',
         '--train-offsets', OUTPUT / 'official-data/tinystories/train_bpe_8192_offsets.npy',
         '--valid', OUTPUT / 'official-data/tinystories/valid_bpe_8192.pickle'])
else:
    print('Prepared data already exists; skip download/preparation.')
print(prepared_manifest.read_text())


In [ ]:
if CACHE_VQ_CODES:
    if not (OUTPUT / 'codes/manifest.json').exists():
        run([sys.executable, '-m', 'baseline_data.encode_codes', '--config', CONFIG,
             '--batch-size', '32'])
    else:
        print('VQ code cache already exists; skip encoding.')
    print((OUTPUT / 'codes/manifest.json').read_text())
else:
    print('Training notebook will need to generate codes once before NCM training.')


In [ ]:
# Check content hashes before saving reusable outputs or removing raw downloads.
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from training.runtime import load_config, json_write, environment
from training.train import verify_data
verify_data(load_config(CONFIG), with_codes=CACHE_VQ_CODES)

lock = Path('/kaggle/working/ncm-environment.txt')
if lock.exists():
    (OUTPUT / 'environment.txt').write_text(lock.read_text())
json_write({'environment': environment(), 'code_cache_included': CACHE_VQ_CODES,
            'tokenizer_policy': 'official pretrained TinyStories VQ-VAE; FP32 assignments',
            'notes': 'Generator architecture changes can reuse codes; tokenizer changes require a new code cache.'},
           OUTPUT / 'preparation-info.json')
(OUTPUT / 'README.md').write_text(
    '# Reusable TinyStories inputs for AR/NCM\n\n'
    'prepared/: common text-token rows and split/BPE provenance.\n'
    'codes/: optional frozen-tokenizer codes and training-only level-zero counts.\n'
    'artifacts/tinystories/: official tokenizer, generator and BPE assets.\n'
    'downloads.json: exact source revisions.\n'
    'VQ codes are tied to the tokenizer checkpoint, not to the NCM generator.\n')
print('Hashes verified.')


In [ ]:
import shutil
if not KEEP_RAW_DOWNLOADS:
    raw = OUTPUT / 'official-data'
    if raw.exists():
        shutil.rmtree(raw)  # Only this notebook's redundant downloaded source folder.
for item in sorted(OUTPUT.iterdir()):
    print(item.name + ('/' if item.is_dir() else ''))
total = sum(p.stat().st_size for p in OUTPUT.rglob('*') if p.is_file())
print(f'Reusable files: {total / 2**30:.2f} GiB at {OUTPUT}')


## Save once, attach to future experiments

1. Save a Kaggle notebook version **with outputs** after all cells succeed.
2. From the saved output, create a Kaggle Dataset containing `ncm-tinystories/`.
3. Set its visibility to **Private**.
4. In `02_train_tinystories_kaggle.ipynb`, add that Dataset through Kaggle's input
   attachment interface and set `DATASET_ROOT` to the folder containing `prepared/`.

Future training notebooks read these inputs directly from `/kaggle/input/`.
TinyStories is not downloaded again. Keep the prepared text rows even when replacing
or retraining the VQ tokenizer; only the corresponding code cache must be rebuilt.
